# 03 - Baseline Machine Learning Models (GLORIA -> Sentinel-2 Proxies)

**Project:** `miniproject2026` — Spatio-Temporal Satellite Water Contamination Analysis & Illegal Dumping Detection Engine  
**Stage:** Day 2 — Task 1 (Dataset Verification & Baseline ML Notebook Setup)  
**Objective:** Prepare reproducible, leakage-resistant baseline regression pipelines for **Chlorophyll-a (`Chla`)** and **Total Suspended Solids (`TSS`)** using the quality-controlled GLORIA training dataset (`data/processed/gloria_cleaned.csv`), and verify compatibility with the multi-lake Sentinel-2 L2A time-series (`data/processed/sentinel_timeseries.csv`).

> **Setup Note:** This notebook replaces the Day 0 classification stub (`RandomForestClassifier` / `features_clean.csv`) with the Day 2 dual-target regression pipeline. Model training, cross-validation, and Sentinel-2 inference cells are prepared below and intentionally **not executed** until the schema mapping proposal in Section 2 is reviewed and approved.

## 1. Imports and Configuration

Import core numerical, plotting, serialization, and scikit-learn modeling utilities, detect the repository root reliably, and fix random seeds for reproducibility.

In [1]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Reproducibility & split configuration
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
EPS = 1e-6

# Plot defaults
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 120


def find_project_root(start_path: Path | None = None) -> Path:
    """Locate the repository root containing SPEC.md and data/processed."""
    current = (start_path or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "SPEC.md").is_file() and (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root containing SPEC.md and data/processed from {current}"
    )


PROJECT_ROOT = find_project_root()
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

GLORIA_PATH = DATA_PROCESSED_DIR / "gloria_cleaned.csv"
SENTINEL_PATH = DATA_PROCESSED_DIR / "sentinel_timeseries.csv"

EXPECTED_LAKE_IDS = ["bellandur", "hussain_sagar", "yamuna_okhla", "sukhna"]

print(f"Project Root : {PROJECT_ROOT}")
print(f"GLORIA Path  : {GLORIA_PATH}")
print(f"Sentinel Path: {SENTINEL_PATH}")
print(f"Models Dir   : {MODELS_DIR}")
print(
    f"Versions     : numpy={np.__version__}, pandas={pd.__version__}, "
    f"sklearn={sklearn.__version__}, scipy={scipy.__version__}, joblib={joblib.__version__}"
)

Project Root : C:\Users\somya\OneDrive\Desktop\miniproject
GLORIA Path  : C:\Users\somya\OneDrive\Desktop\miniproject\data\processed\gloria_cleaned.csv
Sentinel Path: C:\Users\somya\OneDrive\Desktop\miniproject\data\processed\sentinel_timeseries.csv
Models Dir   : C:\Users\somya\OneDrive\Desktop\miniproject\models
Versions     : numpy=2.4.6, pandas=3.1.1, sklearn=1.9.1, scipy=1.17.1, joblib=1.6.0


## 2. Data Loading & Schema Reconciliation

### Dataset Filename & Schema Findings

1. **Filename Reconciliation:**
   - Expected Day 2 filenames `data/processed/gloria_s2_ready.csv` and `data/processed/sentinel_raw_timeseries.csv` are **not** present in the repository.
   - The actual Day 1 outputs present on disk are:
     - `data/processed/gloria_cleaned.csv` (from `ml/01_gloria_eda.ipynb`, 5,008 rows × 62 columns)
     - `data/processed/sentinel_timeseries.csv` (from `ml/02_sentinel_fetch.ipynb`, 128 rows × 36 columns, passing all 13 Day-1 validation checks)
   - Per Day 2 instructions, neither file is renamed or duplicated with dummy CSVs.

2. **GLORIA ↔ Sentinel-2 Schema Mapping Proposal (Pending Approval Before Training):**

| Intended Feature / Target | GLORIA (`gloria_cleaned.csv`) Candidate | Sentinel-2 (`sentinel_timeseries.csv`) | Compatibility & Missingness Notes |
| :--- | :--- | :--- | :--- |
| `B02` (490 nm) | `rho_B2_srf` (or `rho_B2`) | `B02_med` | 0% missing in GLORIA (`rho = pi * Rrs` matches L2A BOA reflectance) |
| `B03` (560 nm) | `rho_B3_srf` (or `rho_B3`) | `B03_med` | 0% missing in GLORIA |
| `B04` (665 nm) | `rho_B4_srf` (or `rho_B4`) | `B04_med` | 0% missing in GLORIA (5 tiny negative dark-water values down to `-0.000219`) |
| `B05` (705 nm) | `rho_B5_srf` (or `rho_B5`) | `B05_med` | `rho_B5_srf` 0% missing; nominal `rho_B5` 7.93% missing |
| `B06` (740 nm) | `rho_B6_srf` (or `rho_B6`) | `B06_med` | ~8.07%–8.09% missing in GLORIA |
| `B07` (783 nm) | `rho_B7_srf` (or `rho_B7`) | `B07_med` | `rho_B7_srf` 17.67% missing; nominal `rho_B7` 21.57% missing |
| `B08` (842 nm) | `rho_B8_srf` (21.87% missing) / `rho_B8` (57.45% missing) | **Missing** (`B8A_med` exported instead) | **Blocker:** `sentinel_timeseries.csv` has `B8A_med`, not `B08_med`; in GLORIA `rho_B8A`/`rho_B8A_srf` are 68.63% missing |
| `ndwi` | `NDWI_B8` (57.65% missing) / `NDWI_B8A` (68.87% missing) | `NDWI_med`, `NDWI_B8A_med` | High GLORIA missingness (>57%); exclude from mandatory `dropna()` features |
| `ndci` | `NDCI_srf` (0.54% missing) / `NDCI` (8.47% missing) | `NDCI_med` | Compatible definition: `(B05 - B04) / (B05 + B04)` |
| `ndti` | `NDTI_srf` (0.10% missing) / `NDTI` (0.10% missing) | `NDTI_med` | Compatible definition: `(B04 - B03) / (B04 + B03)` |
| `chla`, `tss` | `Chla` (3,746 valid), `TSS` (3,460 valid) | N/A (inference targets) | Keep independent target subsets (`has_valid_chla`, `has_valid_tss`) |

The cell below loads both datasets, verifies required files and core columns, and prints dataset diagnostics. **GLORIA (training) and Sentinel-2 (Indian lake inference) are strictly separated and never concatenated.**

In [2]:
for required_path in [GLORIA_PATH, SENTINEL_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(f"Required dataset missing: {required_path}")

gloria_df = pd.read_csv(GLORIA_PATH)
sentinel_df = pd.read_csv(SENTINEL_PATH)

# Validate required core columns in actual datasets
required_gloria_cols = [
    "GLORIA_ID", "group_key", "Latitude", "Longitude",
    "Chla", "TSS", "has_valid_chla", "has_valid_tss",
    "rho_B2_srf", "rho_B3_srf", "rho_B4_srf", "rho_B5_srf", "rho_B6_srf", "rho_B7_srf",
    "NDCI_srf", "NDTI_srf",
]
required_sentinel_cols = [
    "lake_id", "date", "scene_id", "quality_flag", "selected_water_px",
    "B02_med", "B03_med", "B04_med", "B05_med", "B06_med", "B07_med", "B8A_med",
    "NDWI_med", "NDCI_med", "NDTI_med",
]

missing_gloria = [c for c in required_gloria_cols if c not in gloria_df.columns]
missing_sentinel = [c for c in required_sentinel_cols if c not in sentinel_df.columns]
if missing_gloria:
    raise ValueError(f"GLORIA dataset missing expected columns: {missing_gloria}")
if missing_sentinel:
    raise ValueError(f"Sentinel dataset missing expected columns: {missing_sentinel}")

print(f"GLORIA shape   : {gloria_df.shape} | duplicates: {gloria_df.duplicated().sum()}")
print(f"Sentinel shape : {sentinel_df.shape} | duplicates: {sentinel_df.duplicated().sum()}")
print("\nGLORIA columns (62):", gloria_df.columns.tolist())
print("\nSentinel columns (36):", sentinel_df.columns.tolist())

print("\n--- GLORIA Target & Key Spectral Missingness ---")
audit_cols = [
    "Chla", "TSS", "group_key", "Latitude", "Longitude",
    "rho_B2_srf", "rho_B3_srf", "rho_B4_srf", "rho_B5_srf",
    "rho_B6_srf", "rho_B7_srf", "rho_B8_srf", "rho_B8A_srf",
    "NDCI_srf", "NDTI_srf", "NDWI_B8", "NDWI_B8A",
]
miss_summary = pd.DataFrame({
    "dtype": gloria_df[audit_cols].dtypes.astype(str),
    "non_null": gloria_df[audit_cols].notna().sum(),
    "missing": gloria_df[audit_cols].isna().sum(),
    "missing_pct": (gloria_df[audit_cols].isna().mean() * 100).round(2),
    "min": gloria_df[audit_cols].min(numeric_only=True),
    "max": gloria_df[audit_cols].max(numeric_only=True),
})
display(miss_summary)

print("\n--- Sentinel-2 Lake & Quality Summary ---")
print("Lakes       :", sentinel_df["lake_id"].value_counts().to_dict())
print("Date range  :", sentinel_df["date"].min(), "to", sentinel_df["date"].max())
print("Quality flag:", sentinel_df["quality_flag"].value_counts().to_dict())
print("Dup (lake_id, date):", sentinel_df.duplicated(subset=["lake_id", "date"]).sum())
print("Total NaN in Sentinel CSV:", int(sentinel_df.isna().sum().sum()))

print("\nGLORIA sample (first 3 rows):")
display(gloria_df[["GLORIA_ID", "group_key", "Chla", "TSS", "rho_B2_srf", "rho_B3_srf", "rho_B4_srf", "rho_B5_srf", "NDCI_srf", "NDTI_srf"]].head(3))
print("\nSentinel-2 sample (first 3 rows):")
display(sentinel_df[["lake_id", "date", "quality_flag", "B02_med", "B03_med", "B04_med", "B05_med", "B8A_med", "NDWI_med", "NDCI_med", "NDTI_med"]].head(3))

GLORIA shape   : (5008, 62) | duplicates: 0
Sentinel shape : (128, 36) | duplicates: 0

GLORIA columns (62): ['GLORIA_ID', 'Dataset_ID', 'Site_name', 'group_key', 'Country', 'Country_code', 'Latitude', 'Longitude', 'Date_Time_UTC', 'Water_body_type', 'Water_type', 'Chla', 'TSS', 'Turbidity', 'Secchi_depth', 'aCDOM440', 'has_valid_chla', 'has_valid_tss', 'has_both_targets', 'Rrs_B1', 'Rrs_B2', 'Rrs_B3', 'Rrs_B4', 'Rrs_B5', 'Rrs_B6', 'Rrs_B7', 'Rrs_B8', 'Rrs_B8A', 'rho_B1', 'rho_B2', 'rho_B3', 'rho_B4', 'rho_B5', 'rho_B6', 'rho_B7', 'rho_B8', 'rho_B8A', 'Rrs_B1_srf', 'Rrs_B2_srf', 'Rrs_B3_srf', 'Rrs_B4_srf', 'Rrs_B5_srf', 'Rrs_B6_srf', 'Rrs_B7_srf', 'Rrs_B8_srf', 'Rrs_B8A_srf', 'rho_B1_srf', 'rho_B2_srf', 'rho_B3_srf', 'rho_B4_srf', 'rho_B5_srf', 'rho_B6_srf', 'rho_B7_srf', 'rho_B8_srf', 'rho_B8A_srf', 'NDCI', 'NDCI_srf', 'NDTI', 'NDTI_srf', 'NDWI_B8A', 'NDWI_B8', '2BDA']

Sentinel columns (36): ['lake_id', 'date', 'scene_id', 'processing_baseline', 'tile_cloud_pct', 'local_cloud_pct', '

,dtype,non_null,missing,missing_pct,min,max
Chla,float64,3746,1262,25.20,0.000000,13296.700000
Latitude,float64,4961,47,0.94,-45.996582,66.433070
Longitude,float64,4961,47,0.94,-122.227300,177.835552
NDCI_srf,float64,4981,27,0.54,-0.935458,0.938208
NDTI_srf,float64,5003,5,0.10,-0.985221,0.512942
NDWI_B8,float64,2121,2887,57.65,-0.644104,0.997360
NDWI_B8A,float64,1559,3449,68.87,-0.636921,0.998891
TSS,float64,3460,1548,30.91,0.000000,2626.820000
group_key,str,5008,0,0.00,NaN,NaN
rho_B2_srf,float64,5008,0,0.00,0.000067,0.191787



--- Sentinel-2 Lake & Quality Summary ---
Lakes       : {'bellandur': 32, 'hussain_sagar': 32, 'yamuna_okhla': 32, 'sukhna': 32}
Date range  : 2024-06-01 to 2026-11-24
Quality flag: {'PASS_ERODED': 128}
Dup (lake_id, date): 0
Total NaN in Sentinel CSV: 0

GLORIA sample (first 3 rows):


,GLORIA_ID,group_key,Chla,TSS,rho_B2_srf,rho_B3_srf,rho_B4_srf,rho_B5_srf,NDCI_srf,NDTI_srf
0,GID_1,AlikasK_EE_UT-TO_Lake Peipsi,NaN,2.67,0.005318,0.009979,0.007693,0.006647,-0.072941,-0.129346
1,GID_2,AlikasK_EE_UT-TO_Lake Peipsi,NaN,5.67,0.005559,0.010674,0.008106,0.007014,-0.072207,-0.136776
2,GID_3,AlikasK_EE_UT-TO_Lake Peipsi,NaN,8.00,0.006297,0.011726,0.009017,0.007955,-0.062586,-0.130587



Sentinel-2 sample (first 3 rows):


,lake_id,date,quality_flag,B02_med,B03_med,B04_med,B05_med,B8A_med,NDWI_med,NDCI_med,NDTI_med
0,bellandur,2024-06-01,PASS_ERODED,0.0700,0.0974,0.0771,0.126406,0.121450,-0.110724,0.209907,-0.094552
1,bellandur,2024-11-08,PASS_ERODED,0.0396,0.0452,0.0317,0.044656,0.037144,0.133550,0.137001,-0.164179
2,bellandur,2025-01-07,PASS_ERODED,0.0340,0.0500,0.0300,0.031644,0.018322,0.347879,0.058742,-0.232195


## 3. Feature Engineering

The initial methodology specifies bands `B02–B08`, `NDWI`, `NDCI`, `NDTI`, and band ratios `B03/B02` and `B04/B03`.

Based on the Section 2 audit:
- **Reflectance scale consistency:** GLORIA `rho_B*` / `rho_B*_srf` columns (`rho = pi * Rrs`) are already in dimensionless surface reflectance units matching Sentinel-2 L2A reflectance (`uint16 / 10000.0`). No unit rescaling is applied.
- **Missing-band guard:** `rho_B8A` / `rho_B8A_srf` (68.63% missing) and `NDWI_B8` / `NDWI_B8A` (>57% missing) are **excluded** from mandatory `dropna()` feature sets so valid observations are not silently discarded. Furthermore, `sentinel_timeseries.csv` exports `B8A_med` rather than `B08_med`, so a harmonized cross-dataset band list (`B02–B07` + `NDCI` + `NDTI` + `B03/B02` + `B04/B03`, optionally adding `B08` only after schema sign-off) must be confirmed before training.
- **No target leakage:** `Chla`, `TSS`, `Turbidity`, `Secchi_depth`, and `aCDOM440` are strictly excluded from feature matrices.

The helper functions below prepare aligned GLORIA and Sentinel-2 feature matrices once the band subset is approved.

In [ ]:
# Harmonized band set excluding B8A (68.6% missing in GLORIA) and B08 (absent from sentinel_timeseries.csv)
APPROVED_CORE_BANDS = ["B02", "B03", "B04", "B05", "B06", "B07"]
FORBIDDEN_TARGET_COLS = {"Chla", "TSS", "Turbidity", "Secchi_depth", "aCDOM440"}


def _safe_band_ratio(num: pd.Series, den: pd.Series, min_den: float = 1e-6) -> pd.Series:
    """Compute num / den safely, masking near-zero or negative denominators to NaN."""
    valid_den = den.where(den > min_den)
    return num / valid_den


def build_gloria_features(df: pd.DataFrame, use_srf: bool = True, include_b08: bool = False) -> pd.DataFrame:
    """Construct spectral features from GLORIA surface reflectance (rho) without target leakage."""
    suffix = "_srf" if use_srf else ""
    bands = list(APPROVED_CORE_BANDS) + (["B08"] if include_b08 else [])

    feat = pd.DataFrame(index=df.index)
    for band in bands:
        # GLORIA uses B2..B8 while Sentinel-2 uses B02..B08
        gloria_band = f"B{int(band[1:])}"
        col_name = f"rho_{gloria_band}{suffix}"
        if col_name not in df.columns:
            raise KeyError(f"Expected GLORIA reflectance column missing: {col_name}")
        feat[band] = df[col_name].astype(float)

    ndci_col = f"NDCI{suffix}"
    ndti_col = f"NDTI{suffix}"
    for col_name in [ndci_col, ndti_col]:
        if col_name not in df.columns:
            raise KeyError(f"Expected GLORIA index column missing: {col_name}")

    feat["NDCI"] = df[ndci_col].astype(float)
    feat["NDTI"] = df[ndti_col].astype(float)
    feat["ratio_B03_B02"] = _safe_band_ratio(feat["B03"], feat["B02"])
    feat["ratio_B04_B03"] = _safe_band_ratio(feat["B04"], feat["B03"])

    assert FORBIDDEN_TARGET_COLS.isdisjoint(feat.columns), "Target leakage detected in features!"
    return feat


def build_sentinel_features(df: pd.DataFrame) -> pd.DataFrame:
    """Construct matching inference features from Sentinel-2 median reflectance columns."""
    feat = pd.DataFrame(index=df.index)
    for band in APPROVED_CORE_BANDS:
        col_name = f"{band}_med"
        if col_name not in df.columns:
            raise KeyError(f"Expected Sentinel-2 column missing: {col_name}")
        feat[band] = df[col_name].astype(float)

    feat["NDCI"] = df["NDCI_med"].astype(float)
    feat["NDTI"] = df["NDTI_med"].astype(float)
    feat["ratio_B03_B02"] = _safe_band_ratio(feat["B03"], feat["B02"])
    feat["ratio_B04_B03"] = _safe_band_ratio(feat["B04"], feat["B03"])
    return feat

## 4. Target Preparation

Prepare **two independent regression datasets** so missing `Chla` measurements do not drop valid `TSS` rows (and vice versa):
- **Model A (Chlorophyll-a):** `y = log1p(Chla)` using valid, finite, non-negative `Chla` observations ($N = 3,746$ before feature completeness filtering).
- **Model B (Total Suspended Solids):** `y = log1p(TSS)` using valid, finite, non-negative `TSS` observations ($N = 3,460$ before feature completeness filtering).

In [ ]:
def prepare_target_dataset(
    df: pd.DataFrame,
    target_col: str,
    feature_df: pd.DataFrame,
    group_col: str = "group_key",
    drop_missing_features: bool = False,
):
    """Filter valid non-negative target rows and return (X, y_log1p, groups, metadata_df)."""
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found.")
    if group_col not in df.columns:
        raise KeyError(f"Grouping column '{group_col}' not found.")

    valid_target_mask = df[target_col].notna() & np.isfinite(df[target_col]) & (df[target_col] >= 0)
    valid_group_mask = df[group_col].notna()
    mask = valid_target_mask & valid_group_mask

    if drop_missing_features:
        mask = mask & feature_df.notna().all(axis=1)

    sub_df = df.loc[mask].copy()
    X = feature_df.loc[mask].copy()
    y_log = np.log1p(sub_df[target_col].astype(float))
    groups = sub_df[group_col].astype(str)

    return X, y_log, groups, sub_df

## 5. Data Splitting Strategy

To prevent spatial and campaign data leakage across observations from the same water body:
- **Held-out test split:** `GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)` grouped by `group_key` (`Dataset_ID + Site_name`, 437 unique groups overall).
- **Cross-validation on training set:** `GroupKFold(n_splits=5)` using the same grouping variable.
- **Spatial coordinate check:** The inspection function explicitly verifies that neither `group_key` nor rounded `(Latitude, Longitude)` pairs appear in both training and test splits. *(Audit note: 1 coordinate pair `(54.30252, 17.61468)` in `FicekD_PL_APSL` spans two `group_key` labels — `Lake Jasień Północny` and `Lake Jasień Południowy` — and 47 rows lack coordinates; `build_spatial_group_key` unifies identical non-null coordinates prior to splitting.)*

In [ ]:
def build_spatial_group_key(df: pd.DataFrame, base_col: str = "group_key") -> pd.Series:
    """Ensure identical rounded (Latitude, Longitude) observations share a single split group."""
    groups = df[base_col].astype(str).copy()
    has_coords = df["Latitude"].notna() & df["Longitude"].notna()
    coord_str = (
        df.loc[has_coords, "Latitude"].round(5).astype(str)
        + "_"
        + df.loc[has_coords, "Longitude"].round(5).astype(str)
    )
    # Map each unique coordinate pair to its first observed group_key
    coord_to_group = (
        pd.DataFrame({"coord": coord_str, "group": groups.loc[has_coords]})
        .drop_duplicates("coord")
        .set_index("coord")["group"]
    )
    groups.loc[has_coords] = coord_str.map(coord_to_group)
    return groups


def make_leakage_free_split(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    meta_df: pd.DataFrame,
    test_size: float = TEST_SIZE,
    n_splits: int = N_SPLITS,
    random_state: int = RANDOM_STATE,
):
    """Create a GroupShuffleSplit train/test partition and GroupKFold CV splitter."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])
    assert train_groups.isdisjoint(test_groups), "Group leakage between train and test!"

    # Verify no identical rounded lat/lon coordinates span train and test
    coords_train = set(
        meta_df.iloc[train_idx].dropna(subset=["Latitude", "Longitude"])
        .apply(lambda r: (round(r["Latitude"], 5), round(r["Longitude"], 5)), axis=1)
    )
    coords_test = set(
        meta_df.iloc[test_idx].dropna(subset=["Latitude", "Longitude"])
        .apply(lambda r: (round(r["Latitude"], 5), round(r["Longitude"], 5)), axis=1)
    )
    coord_overlap = coords_train.intersection(coords_test)
    assert not coord_overlap, f"Coordinate leakage detected across split: {coord_overlap}"

    gkf = GroupKFold(n_splits=n_splits)
    return train_idx, test_idx, gkf

## 6. Regression Model Candidates

Three candidate regression pipelines are configured identically for both targets (`Chla` and `TSS`):
1. **Ridge Regression:** `SimpleImputer(strategy="median")` + `StandardScaler()` + `Ridge(alpha=1.0)`
2. **Random Forest Regressor:** `SimpleImputer(strategy="median")` + `RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42)`
3. **HistGradientBoostingRegressor:** `HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=42)` (handles `NaN` values natively)

All preprocessing steps (`SimpleImputer`, `StandardScaler`) are encapsulated inside scikit-learn `Pipeline` objects so they are fitted **strictly on training folds**.

In [ ]:
def build_candidate_models(random_state: int = RANDOM_STATE) -> dict[str, Pipeline]:
    """Return unfitted baseline regression pipelines for a single water-quality target."""
    return {
        "Ridge": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0, random_state=random_state)),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=3,
                n_jobs=-1,
                random_state=random_state,
            )),
        ]),
        "HistGradientBoosting": Pipeline([
            ("model", HistGradientBoostingRegressor(
                max_iter=300,
                learning_rate=0.05,
                max_leaf_nodes=31,
                random_state=random_state,
            )),
        ]),
    }

## 7. Evaluation

Define evaluation metrics (`R²`, `RMSE`, `MAE`) computed in **both `log1p` space and back-transformed original concentration units** (`expm1`, clipped at `>= 0`), along with a structured comparison table builder across all three candidate models and both targets.

> **Note:** No model performance is claimed until training and `GroupKFold` / held-out evaluation are executed after schema approval.

In [ ]:
def compute_regression_metrics(y_true_log: np.ndarray, y_pred_log: np.ndarray) -> dict[str, float]:
    """Compute R2, RMSE, and MAE in both log1p space and back-transformed original units."""
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.clip(np.expm1(y_pred_log), a_min=0.0, a_max=None)

    return {
        "r2_log": float(r2_score(y_true_log, y_pred_log)),
        "rmse_log": float(root_mean_squared_error(y_true_log, y_pred_log)),
        "mae_log": float(mean_absolute_error(y_true_log, y_pred_log)),
        "r2_orig": float(r2_score(y_true_orig, y_pred_orig)),
        "rmse_orig": float(root_mean_squared_error(y_true_orig, y_pred_orig)),
        "mae_orig": float(mean_absolute_error(y_true_orig, y_pred_orig)),
    }


def format_comparison_table(records: list[dict]) -> pd.DataFrame:
    """Format evaluation records across targets and models into a sorted comparison DataFrame."""
    if not records:
        return pd.DataFrame(
            columns=[
                "target", "model", "split",
                "r2_log", "rmse_log", "mae_log",
                "r2_orig", "rmse_orig", "mae_orig",
            ]
        )
    return pd.DataFrame(records).sort_values(["target", "split", "rmse_log"]).reset_index(drop=True)

## 8. Model Saving Preparation

Reserve the `models/` output directory and define the artifact serialization contract for future trained models:
- `models/chla_proxy.joblib`
- `models/tss_proxy.joblib`
- `models/baseline_metrics.json`

> **Safety Guard:** Untrained or placeholder models are **never** saved, and existing trained model files are never overwritten blindly.

In [ ]:
RESERVED_MODEL_PATHS = {
    "chla_model": MODELS_DIR / "chla_proxy.joblib",
    "tss_model": MODELS_DIR / "tss_proxy.joblib",
    "metrics_json": MODELS_DIR / "baseline_metrics.json",
}


def prepare_models_directory(models_dir: Path = MODELS_DIR) -> list[str]:
    """Create models/ directory if missing and list any pre-existing artifact files."""
    models_dir.mkdir(parents=True, exist_ok=True)
    existing = [
        p.name for p in models_dir.iterdir()
        if p.is_file() and not p.name.startswith(".")
    ]
    return sorted(existing)


def save_trained_artifacts(
    chla_pipeline: Pipeline,
    tss_pipeline: Pipeline,
    metrics_payload: dict,
    overwrite: bool = False,
) -> None:
    """Persist trained Chla/TSS pipelines and evaluation metrics after training."""
    prepare_models_directory(MODELS_DIR)
    for label, path in RESERVED_MODEL_PATHS.items():
        if path.exists() and not overwrite:
            raise FileExistsError(f"Refusing to overwrite existing artifact ({label}): {path}")

    joblib.dump(chla_pipeline, RESERVED_MODEL_PATHS["chla_model"])
    joblib.dump(tss_pipeline, RESERVED_MODEL_PATHS["tss_model"])
    RESERVED_MODEL_PATHS["metrics_json"].write_text(
        json.dumps(metrics_payload, indent=2), encoding="utf-8"
    )